# 07 — Análise de Sub-estruturas: UMAP+HDBSCAN vs LOF

**Objetivo:** Identificar sub-grupos naturais e municípios excepcionais dentro de cada macro-cluster K-Means,  
testando duas abordagens alternativas ao HDBSCAN original e documentando a decisão metodológica.

**Entrada:** `rqual_2022_clusterizado.parquet`  
**Saídas:** `rqual_2022_clusterizado_v2.parquet`, `municipios_excepcionais_lof.csv`

---

## Histórico de tentativas

| Tentativa | Abordagem | Resultado | Problema |
|-----------|-----------|-----------|----------|
| **1** | HDBSCAN direto nas 20 features | 96.3% ruído, 2 sub-clusters | Maldição da dimensionalidade |
| **2** | UMAP (redução dimensional) + HDBSCAN | A avaliar | — |
| **3** | Local Outlier Factor (LOF) | A avaliar | — |

> **Contexto:** Com Silhouette=0.83, o K-Means K=5 captura bem a macro-estrutura. O desafio aqui é  
> identificar *dentro* de cada cluster: (a) sub-grupos naturais e (b) municípios atípicos para uso regulatório.

In [ ]:
# Bloco 0 — Setup e importações
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from IPython.display import display

# Redução dimensional
try:
    import umap
    print(f'umap-learn: OK (versão {umap.__version__})')
except ImportError:
    raise ImportError('Instale umap-learn: pip install umap-learn')

# Clustering de densidade
try:
    import hdbscan
    print(f'hdbscan: OK (versão {hdbscan.__version__})')
except ImportError:
    raise ImportError('Instale hdbscan: pip install hdbscan')

# Detecção de anomalias
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import silhouette_score, silhouette_samples
import sklearn
print(f'scikit-learn: OK (versão {sklearn.__version__})')

RANDOM_STATE = 42
ARQ = Path('rqual_2022_clusterizado.parquet')

CORES_CLUSTERS = ['#1976D2', '#388E3C', '#F57C00', '#D32F2F', '#7B1FA2']
CLUSTER_NOMES = {
    0: 'C0: Urbano-Avançado',
    1: 'C1: Intermediário',
    2: 'C2: Nordeste Periférico',
    3: 'C3: Norte/Amazônico',
    4: 'C4: Capitais/Destaques',
}
print('\nSetup concluído.')

In [ ]:
# Bloco 1 — Carregar e preparar dados
df = pd.read_parquet(ARQ)
print(f'Shape carregado: {df.shape}')
print(f'Colunas: {list(df.columns)}')

# Renomear colunas socioeconômicas longas (mesma lógica do NB06)
rename_map = {}
for col in df.columns:
    cl = col.lower()
    if 'agropecuária' in cl or 'agropecuaria' in cl: rename_map[col] = 'pib_agropecuaria'
    elif 'indústria' in cl or 'industria' in cl:      rename_map[col] = 'pib_industria'
    elif 'per capita' in cl:                           rename_map[col] = 'pib_per_capita'
    elif 'área' in cl or ('area' in cl and 'km' in cl): rename_map[col] = 'area_km2'
    elif 'densidade' in cl:                            rename_map[col] = 'densidade'
    elif 'rural' in cl and 'urb' in cl.upper():        rename_map[col] = 'pop_rural'
    elif col.startswith('IDHM') and '__' in col:       rename_map[col] = 'idhm'
    elif col.startswith('LAT__lat'):                   rename_map[col] = 'lat'
    elif col.startswith('LAT__lon'):                   rename_map[col] = 'lon'
df = df.rename(columns=rename_map)

# Detectar colunas de features: numéricas, excluindo metadados e labels
EXCLUIR = {
    'cod_mun', 'nome_mun_rqual', 'nome_mun', 'uf_rqual', 'uf',
    'kmeans_cluster', 'hdbscan_cluster', 'hdbscan_noise', 'perfil_hdbscan',
    'lat', 'lon', 'regiao', 'umap_x', 'umap_y',
}

FEAT_COLS = [
    c for c in df.select_dtypes(include='number').columns
    if c not in EXCLUIR
    and 'cluster' not in c.lower()
    and 'noise' not in c.lower()
    and 'lof' not in c.lower()
    and 'umap' not in c.lower()
]

# Indicadores RQUAL e socioeconômicas (para tabelas e plots)
IND_COLS = [c for c in ['IND2','IND4','IND5','IND8','IND9','INF1','INF4-UP'] if c in df.columns]
SOC_COLS = [c for c in ['pib_agropecuaria','pib_industria','pib_per_capita',
                         'area_km2','densidade','pop_rural','idhm'] if c in df.columns]

X = df[FEAT_COLS].values

print(f'\nFeatures detectadas ({len(FEAT_COLS)}): {FEAT_COLS}')
print(f'Matriz X: {X.shape} | NaNs: {np.isnan(X).sum()}')
print(f'Indicadores RQUAL: {IND_COLS}')
print(f'\nDistribuição K-Means:')
display(df['kmeans_cluster'].value_counts().sort_index().rename('n_municípios').to_frame())

---
## Seção 1 — Tentativa 1: HDBSCAN Original (Diagnóstico)

**O que foi feito:** HDBSCAN aplicado diretamente no espaço de features (20 dimensões), dentro de cada macro-cluster K-Means.

**Por que falhou:**
- Em alta dimensionalidade, as distâncias euclidianas entre pontos tendem a se **homogeneizar** — a diferença entre o vizinho mais próximo e o mais distante diminui proporcionalmente.
- O HDBSCAN busca regiões de **alta densidade relativa**. Em 20 dimensões, toda a nuvem de pontos parece ter densidade uniforme.
- Resultado: o algoritmo não encontra núcleos densos e classifica a maioria como ruído.

In [ ]:
# Bloco 2 — Recap métricas do HDBSCAN original
n_total      = len(df)
n_ruido_orig = df['hdbscan_noise'].sum()
n_sub_orig   = (~df['hdbscan_noise']).sum()
pct_ruido_orig = 100 * n_ruido_orig / n_total

print('=' * 55)
print('TENTATIVA 1 — HDBSCAN Direto (20 dimensões)')
print('=' * 55)
print(f'  Municípios em sub-clusters : {n_sub_orig:5,} ({100 - pct_ruido_orig:.1f}%)')
print(f'  Municípios como ruído      : {n_ruido_orig:5,} ({pct_ruido_orig:.1f}%)')
print()
print('Sub-clusters encontrados:')
resumo = (
    df[~df['hdbscan_noise']]
    .groupby(['kmeans_cluster', 'hdbscan_cluster'])
    .size()
    .reset_index(name='n')
)
resumo['cluster_nome'] = resumo['kmeans_cluster'].map(CLUSTER_NOMES)
display(resumo[['cluster_nome', 'hdbscan_cluster', 'n']].rename(
    columns={'cluster_nome': 'Macro-Cluster', 'hdbscan_cluster': 'Sub-Cluster', 'n': 'Municípios'}
))
print()
print('Diagnóstico: C1, C2, C4 são completamente ruído. C0 e C3 têm sub-estrutura,')
print('mas 96.3% do total é ruído. Abordagem insatisfatória para uso analítico.')

---
## Seção 2 — Tentativa 2: UMAP + HDBSCAN

**Estratégia:** Antes de aplicar o HDBSCAN, reduzir a dimensionalidade com UMAP:
1. **UMAP 2D** — apenas para visualização, entender a estrutura global
2. **UMAP 10D** — preserva mais informação, usado como entrada para o HDBSCAN
3. **HDBSCAN global** no espaço UMAP 10D (não mais por cluster)
4. **HDBSCAN por cluster** no espaço UMAP 10D (para comparação direta)

**Por que UMAP ajuda o HDBSCAN:**  
O UMAP preserva a estrutura de vizinhança local enquanto coloca pontos similares próximos e pontos diferentes distantes — criando regiões de densidade variável que o HDBSCAN consegue detectar.

In [ ]:
# Bloco A.1 — UMAP 2D (para visualização)
print('Ajustando UMAP 2D para visualização...')
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,       # ligeiramente espalhado: melhor para visualizar separação
    metric='euclidean',
    random_state=RANDOM_STATE,
)
emb_2d = reducer_2d.fit_transform(X)
df['umap_x'] = emb_2d[:, 0]
df['umap_y'] = emb_2d[:, 1]
print(f'Embedding 2D: {emb_2d.shape} ✓')

# Plot: UMAP 2D colorido por cluster K-Means
fig, ax = plt.subplots(figsize=(10, 8))
for k, cor in enumerate(CORES_CLUSTERS):
    mask = df['kmeans_cluster'] == k
    n = mask.sum()
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
               c=cor, s=8, alpha=0.6, linewidths=0,
               label=f'{CLUSTER_NOMES[k]} ({n:,})')
ax.set_title('Projeção UMAP 2D — colorida por cluster K-Means', fontsize=13)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.legend(fontsize=9, markerscale=3, framealpha=0.9)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig('fig_umap_2d_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_umap_2d_kmeans.png')
print()
print('Observação: o UMAP revela se os clusters K-Means são coesos ou se há sobreposição.')
print('Clusters bem separados no UMAP indicam que a segmentação é robusta.')

In [ ]:
# Bloco A.2 — UMAP 10D (para clustering)
# min_dist=0.0 é recomendado quando o embedding é usado para clustering:
# pontos do mesmo grupo ficam mais compactos, criando variação de densidade.
N_UMAP_DIMS = 10
print(f'Ajustando UMAP {N_UMAP_DIMS}D para clustering (min_dist=0.0)...')
reducer_nd = umap.UMAP(
    n_components=N_UMAP_DIMS,
    n_neighbors=15,
    min_dist=0.0,       # compactar clusters: essencial para HDBSCAN
    metric='euclidean',
    random_state=RANDOM_STATE,
)
emb_nd = reducer_nd.fit_transform(X)
print(f'Embedding {N_UMAP_DIMS}D: {emb_nd.shape} ✓')

In [ ]:
# Bloco A.3 — HDBSCAN Global no espaço UMAP 10D
# Aplicar no dataset completo (diferente da tentativa 1, que era por cluster)
hdb_global = hdbscan.HDBSCAN(
    min_cluster_size=30,    # ~0.5% dos 5570 municípios
    min_samples=10,
    cluster_selection_method='eom',  # Excess of Mass: máxima estabilidade
    metric='euclidean',
)
labels_global = hdb_global.fit_predict(emb_nd)
df['umap_hdbscan_global'] = labels_global

n_clusters_g  = len(set(labels_global)) - (1 if -1 in labels_global else 0)
n_ruido_g     = (labels_global == -1).sum()
pct_ruido_g   = 100 * n_ruido_g / len(df)

print('=' * 55)
print(f'TENTATIVA 2A — UMAP {N_UMAP_DIMS}D + HDBSCAN Global')
print('=' * 55)
print(f'  Sub-clusters encontrados   : {n_clusters_g}')
print(f'  Municípios em sub-clusters : {(labels_global != -1).sum():5,} ({100 - pct_ruido_g:.1f}%)')
print(f'  Municípios como ruído      : {n_ruido_g:5,} ({pct_ruido_g:.1f}%)')
print()
print('Distribuição por sub-cluster:')
dist_global = pd.Series(labels_global).value_counts().sort_index()
dist_global.index = dist_global.index.map(lambda x: 'Ruído' if x == -1 else f'Sub-{x}')
display(dist_global.rename('n_municípios').to_frame())

# Verificar como os sub-clusters UMAP+HDBSCAN se mapeiam nos clusters K-Means
print('\nMapeamento sub-clusters UMAP+HDBSCAN × macro-clusters K-Means:')
cross = pd.crosstab(
    df['umap_hdbscan_global'].map(lambda x: 'Ruído' if x == -1 else f'Sub-{x}'),
    df['kmeans_cluster'].map(CLUSTER_NOMES)
)
display(cross)

In [ ]:
# Bloco A.4 — HDBSCAN por cluster no espaço UMAP 10D
# Abordagem equivalente à tentativa 1, mas agora com dimensionalidade reduzida
print('TENTATIVA 2B — UMAP 10D + HDBSCAN por macro-cluster')
print('=' * 55)

df['umap_hdbscan_local'] = -1  # inicializar como ruído
metricas_hdb_local = {}

idx_array = np.arange(len(df))

for k in sorted(df['kmeans_cluster'].unique()):
    mask = (df['kmeans_cluster'] == k).values
    n = mask.sum()
    X_sub = emb_nd[mask]

    # min_cluster_size proporcional: mínimo 10, máximo 5% do cluster
    min_cs = max(10, int(0.05 * n))

    hdb = hdbscan.HDBSCAN(
        min_cluster_size=min_cs,
        min_samples=5,
        cluster_selection_method='eom',
        metric='euclidean',
    )
    labels = hdb.fit_predict(X_sub)

    # Atribuir label único (prefixo K para evitar colisão entre clusters)
    labels_unique = np.where(labels == -1, -1, k * 1000 + labels)
    df.loc[df['kmeans_cluster'] == k, 'umap_hdbscan_local'] = labels_unique

    n_sub      = len(set(labels)) - (1 if -1 in labels else 0)
    n_ruid     = (labels == -1).sum()
    pct_ruid   = 100 * n_ruid / n

    metricas_hdb_local[k] = {
        'n_total': n, 'n_sub_clusters': n_sub,
        'n_ruido': n_ruid, 'pct_ruido': pct_ruid
    }
    print(f'  C{k} ({n:4,} mun.) | min_cluster_size={min_cs:3d} '
          f'| Sub-clusters: {n_sub} | Ruído: {pct_ruid:.1f}%')

pct_ruido_local_media = np.mean([v['pct_ruido'] for v in metricas_hdb_local.values()])
# Ponderada pelo tamanho do cluster
total_ruido_local = sum(v['n_ruido'] for v in metricas_hdb_local.values())
pct_ruido_local_ponderada = 100 * total_ruido_local / len(df)

print()
print(f'  Ruído ponderado (total):     {pct_ruido_local_ponderada:.1f}%')
print()
print('Comparativo de ruído:')
print(f'  Tentativa 1 (HDBSCAN direto 20D) : 96.3%')
print(f'  Tentativa 2A (UMAP 10D + global)  : {pct_ruido_g:.1f}%')
print(f'  Tentativa 2B (UMAP 10D + local)   : {pct_ruido_local_ponderada:.1f}%')

In [ ]:
# Bloco A.5 — Visualização dos resultados UMAP+HDBSCAN
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# — Painel 1: K-Means original no espaço UMAP
ax = axes[0]
for k, cor in enumerate(CORES_CLUSTERS):
    mask = df['kmeans_cluster'] == k
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
               c=cor, s=6, alpha=0.5, linewidths=0, label=f'C{k}')
ax.set_title('K-Means K=5 (referência)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9, markerscale=2)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

# — Painel 2: UMAP+HDBSCAN global
ax = axes[1]
ids_global = sorted(set(labels_global))
cmap_g = plt.cm.tab20(np.linspace(0, 1, max(len(ids_global), 2)))
noise_mask = df['umap_hdbscan_global'] == -1
ax.scatter(df.loc[noise_mask, 'umap_x'], df.loc[noise_mask, 'umap_y'],
           c='#CCCCCC', s=4, alpha=0.25, linewidths=0, label=f'Ruído ({noise_mask.sum():,})')
ci = 0
for cl in [x for x in ids_global if x != -1]:
    mask = df['umap_hdbscan_global'] == cl
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
               c=[cmap_g[ci]], s=12, alpha=0.85, linewidths=0, label=f'Sub-{cl} ({mask.sum()})')
    ci += 1
ax.set_title(f'UMAP+HDBSCAN Global\n{n_clusters_g} sub-clusters | {pct_ruido_g:.1f}% ruído',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, markerscale=2)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

# — Painel 3: UMAP+HDBSCAN por cluster
ax = axes[2]
noise_local = df['umap_hdbscan_local'] == -1
ax.scatter(df.loc[noise_local, 'umap_x'], df.loc[noise_local, 'umap_y'],
           c='#CCCCCC', s=4, alpha=0.25, linewidths=0, label=f'Ruído ({noise_local.sum():,})')
ids_local = sorted([x for x in df['umap_hdbscan_local'].unique() if x != -1])
cmap_l = plt.cm.tab20(np.linspace(0, 1, max(len(ids_local), 2)))
for i, cl in enumerate(ids_local):
    mask = df['umap_hdbscan_local'] == cl
    k_orig = cl // 1000
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
               c=[cmap_l[i]], s=12, alpha=0.85, linewidths=0,
               label=f'C{k_orig}-Sub{cl%1000} ({mask.sum()})')
ax.set_title(f'UMAP+HDBSCAN Local (por cluster)\n{pct_ruido_local_ponderada:.1f}% ruído',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, markerscale=2, ncol=2)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

plt.suptitle('Tentativa 2: UMAP + HDBSCAN — Comparação global vs local', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_umap_hdbscan_resultado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_umap_hdbscan_resultado.png')

---
## Seção 3 — Tentativa 3: Local Outlier Factor (LOF)

**Estratégia:** Em vez de buscar sub-grupos, o LOF avalia **cada município individualmente** dentro do seu cluster:
- Compara a densidade local do ponto com a de seus vizinhos
- Municípios em regiões menos densas que seus vizinhos → **score alto → excepcionais**
- Produz um **score contínuo** (não binário), ideal para ranking regulatório

**Diferença de objetivo:**  
- HDBSCAN → "existem *grupos* distintos?"
- LOF → "quais *municípios* não se encaixam no padrão do seu cluster?"

**Parâmetros:**
- `n_neighbors=20`: vizinhança local suficiente para 5.570 pontos
- `contamination=0.10`: assume que 10% dos municípios são excepcionais por cluster

In [ ]:
# Bloco B.1 — LOF por macro-cluster
N_NEIGHBORS_LOF = 20
CONTAMINATION   = 0.10

df['lof_score']   = 0.0   # score de anomalia: maior = mais excepcional
df['lof_outlier'] = False  # flag: True = excepcional pelo threshold

print(f'Aplicando LOF por cluster (n_neighbors={N_NEIGHBORS_LOF}, contamination={CONTAMINATION})')
print('=' * 55)

metricas_lof = {}
for k in sorted(df['kmeans_cluster'].unique()):
    idx  = df[df['kmeans_cluster'] == k].index
    n    = len(idx)
    X_sub = df.loc[idx, FEAT_COLS].values

    n_viz = min(N_NEIGHBORS_LOF, n - 1)  # garante n_neighbors < n_samples
    lof = LocalOutlierFactor(
        n_neighbors=n_viz,
        contamination=CONTAMINATION,
        metric='euclidean',
        algorithm='auto',
    )
    pred   = lof.fit_predict(X_sub)           # 1=normal, -1=outlier
    scores = -lof.negative_outlier_factor_     # negar: maior = mais anômalo

    df.loc[idx, 'lof_score']   = scores
    df.loc[idx, 'lof_outlier'] = (pred == -1)

    n_out  = (pred == -1).sum()
    metricas_lof[k] = {
        'n_total': n, 'n_outliers': n_out,
        'pct_outliers': 100 * n_out / n,
        'score_medio': scores[pred == -1].mean() if n_out > 0 else 0,
        'score_max': scores.max()
    }
    print(f'  C{k} ({n:4,} mun.) | Excepcionais: {n_out:4d} ({100*n_out/n:.1f}%) '
          f'| Score médio dos exc.: {metricas_lof[k]["score_medio"]:.2f}')

n_total_outliers = df['lof_outlier'].sum()
print()
print(f'Total de municípios excepcionais: {n_total_outliers} ({100*n_total_outliers/len(df):.1f}%)')

In [ ]:
# Bloco B.2 — Top municípios excepcionais por cluster
print('Top 5 municípios mais excepcionais por cluster (maior LOF score):')
print('(Excepcional = perfil mais distante da densidade típica do seu grupo)\n')

NOMES_CURTOS = {
    0: 'Urbano-Avançado',
    1: 'Intermediário',
    2: 'Nordeste Periférico',
    3: 'Norte/Amazônico',
    4: 'Capitais/Destaques',
}

cols_exib = ['nome_mun_rqual', 'uf_rqual', 'lof_score'] + IND_COLS[:5]
cols_exib = [c for c in cols_exib if c in df.columns]

for k in sorted(df['kmeans_cluster'].unique()):
    sub = df[df['kmeans_cluster'] == k].copy()
    top = sub.nlargest(5, 'lof_score')[cols_exib].reset_index(drop=True)
    top['lof_score'] = top['lof_score'].round(3)
    for c in IND_COLS[:5]:
        if c in top.columns:
            top[c] = top[c].round(3)
    print(f'─── C{k}: {NOMES_CURTOS[k]} ───')
    display(top)
    print()

In [ ]:
# Bloco B.3 — Visualizações LOF
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

# — Painel 1: LOF score no espaço UMAP (mapa de anomalia contínuo)
ax = axes[0]
sc = ax.scatter(
    df['umap_x'], df['umap_y'],
    c=df['lof_score'],
    cmap='YlOrRd',
    norm=mcolors.PowerNorm(gamma=0.4, vmin=df['lof_score'].quantile(0.01),
                            vmax=df['lof_score'].quantile(0.995)),
    s=8, alpha=0.7, linewidths=0,
)
plt.colorbar(sc, ax=ax, label='LOF Score (maior = mais excepcional)')
ax.set_title('Score de Anomalia LOF\n(espaço UMAP 2D)', fontsize=12, fontweight='bold')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

# — Painel 2: Distribuição do LOF score por cluster
ax = axes[1]
for k, cor in enumerate(CORES_CLUSTERS):
    sub = df[df['kmeans_cluster'] == k]['lof_score']
    ax.hist(sub, bins=50, color=cor, alpha=0.55, density=True,
            label=f'C{k} ({len(sub):,})')
ax.axvline(1.5, color='#333333', linestyle='--', alpha=0.8, linewidth=1.5,
           label='Score = 1.5 (referência)')
ax.set_xlabel('LOF Score (maior = mais excepcional)', fontsize=11)
ax.set_ylabel('Densidade', fontsize=11)
ax.set_title('Distribuição do Score LOF por cluster', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0.5, df['lof_score'].quantile(0.995))
ax.grid(alpha=0.2)

plt.suptitle('Tentativa 3: Local Outlier Factor (LOF)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_lof_resultado.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_lof_resultado.png')

In [ ]:
# Bloco B.4 — Mapa geográfico dos municípios excepcionais (LOF)
# Tenta usar coordenadas reais via geobr; fallback para UMAP
try:
    from geobr import read_municipality
    gdf = read_municipality(code_muni='all', year=2020)
    gdf_proj = gdf.to_crs('EPSG:5880')
    gdf['lat_geo'] = gdf_proj.geometry.centroid.to_crs('EPSG:4326').y
    gdf['lon_geo'] = gdf_proj.geometry.centroid.to_crs('EPSG:4326').x
    gdf = gdf.rename(columns={'code_muni': 'cod_mun'})[['cod_mun', 'lat_geo', 'lon_geo']]
    gdf['cod_mun'] = gdf['cod_mun'].astype(int)
    df_map = df.merge(gdf, on='cod_mun', how='left')
    usar_geo = df_map[['lat_geo', 'lon_geo']].notna().all(axis=1).mean() > 0.9
    if usar_geo:
        lat_col, lon_col = 'lat_geo', 'lon_geo'
        titulo_mapa = 'Municípios Excepcionais (LOF)\nDistribuição Geográfica'
    else:
        usar_geo = False
except Exception:
    usar_geo = False

if not usar_geo:
    df_map = df.copy()
    df_map['lat_geo'] = df['umap_y']
    df_map['lon_geo'] = df['umap_x']
    lat_col, lon_col = 'lat_geo', 'lon_geo'
    titulo_mapa = 'Municípios Excepcionais (LOF)\nEspaço UMAP 2D (geobr indisponível)'

fig, ax = plt.subplots(figsize=(11, 12 if usar_geo else 9))

# Inliers: cinza claro
inl = df_map[~df_map['lof_outlier']]
ax.scatter(inl[lon_col], inl[lat_col], c='#DDDDDD', s=3, alpha=0.3, linewidths=0)

# Outliers por cluster, tamanho proporcional ao score
out = df_map[df_map['lof_outlier']].copy()
s_min, s_max = 20, 200
score_norm = (out['lof_score'] - out['lof_score'].min()) / (out['lof_score'].max() - out['lof_score'].min() + 1e-9)
out['marker_size'] = s_min + score_norm * (s_max - s_min)

for k, cor in enumerate(CORES_CLUSTERS):
    sub = out[out['kmeans_cluster'] == k]
    if len(sub) > 0:
        ax.scatter(sub[lon_col], sub[lat_col],
                   c=cor, s=sub['marker_size'], alpha=0.85,
                   linewidths=0.4, edgecolors='white',
                   label=f'{NOMES_CURTOS[k]} ({len(sub)})')

if usar_geo:
    ax.set_xlim(-75, -28)
    ax.set_ylim(-35, 6)
    ax.set_aspect('equal')

ax.set_title(titulo_mapa + '\n(tamanho do ponto ∝ LOF score)', fontsize=12)
ax.legend(fontsize=9, markerscale=1, loc='lower left')
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig('fig_lof_mapa_excepcionais.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_lof_mapa_excepcionais.png')

---
## Seção 4 — Comparação das Abordagens

Avaliamos as três tentativas em cinco critérios:

| Critério | Peso | Tentativa 1: HDBSCAN Direto | Tentativa 2: UMAP+HDBSCAN | Tentativa 3: LOF |
|---|---:|---|---|---|
| **Taxa de ruído** | 25% | 96.3% ❌ | Ver abaixo | N/A (contínuo) ✓ |
| **Sub-grupos detectáveis** | 20% | Apenas C0 e C3 ❌ | Múltiplos clusters ✓ | Não é objetivo |
| **Municípios excepcionais** | 25% | Limitado ❌ | Indireto ⚠️ | Score por município ✓ |
| **Interpretabilidade regulatória** | 20% | Baixa ❌ | Média ⚠️ | Alta ✓ |
| **Robustez a parâmetros** | 10% | Baixa ❌ | Média ⚠️ | Alta ✓ |

In [ ]:
# Bloco C.1 — Scorecard numérico comparativo

# Calcular Silhouette Score dos sub-clusters UMAP+HDBSCAN (global)
# apenas nos pontos não-ruído
mask_nao_ruido_g = labels_global != -1
sil_umap_global = None
if mask_nao_ruido_g.sum() > 10 and len(set(labels_global[mask_nao_ruido_g])) > 1:
    sil_umap_global = silhouette_score(
        emb_nd[mask_nao_ruido_g], labels_global[mask_nao_ruido_g]
    )

print('=' * 65)
print('SCORECARD COMPARATIVO')
print('=' * 65)
print(f'{'Métrica':<40} {'T1':>8} {'T2A':>8} {'T3':>10}')
print('-' * 65)
print(f'{'% ruído (global)':<40} {pct_ruido_orig:>7.1f}% {pct_ruido_g:>7.1f}% {"N/A":>10}')
print(f'{'% ruído (por cluster)':<40} {pct_ruido_orig:>7.1f}% {pct_ruido_local_ponderada:>7.1f}% {"N/A":>10}')
print(f'{'Sub-clusters encontrados':<40} {2:>8} {n_clusters_g:>8} {"N/A":>10}')
print(f'{'Municípios classificados (%)':<40} {100-pct_ruido_orig:>7.1f}% {100-pct_ruido_g:>7.1f}% {"100.0%":>10}')
sil_str = f'{sil_umap_global:.3f}' if sil_umap_global else 'N/A'
print(f'{'Silhouette sub-clusters (não-ruído)':<40} {"—":>8} {sil_str:>8} {"—":>10}')
print(f'{'Outliers excepcionais identificados':<40} {"204":>8} {"N/A":>8} {n_total_outliers:>10}')
print(f'{'Score contínuo por município':<40} {"Não":>8} {"Não":>8} {"Sim":>10}')
print(f'{'Aplicação regulatória (ranking)':<40} {"Baixa":>8} {"Média":>8} {"Alta":>10}')
print('=' * 65)

In [ ]:
# Bloco C.2 — Gráfico comparativo das três tentativas
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Painel 1: T1 — HDBSCAN original no espaço UMAP
ax = axes[0]
noise_orig = df['hdbscan_noise']
ax.scatter(df.loc[noise_orig, 'umap_x'], df.loc[noise_orig, 'umap_y'],
           c='#CCCCCC', s=4, alpha=0.25, linewidths=0, label=f'Ruído ({noise_orig.sum():,})')
ids_orig = sorted(df.loc[~noise_orig, 'hdbscan_cluster'].unique())
cmap_o   = ['#E65100', '#1A237E', '#1B5E20']
for i, cl in enumerate(ids_orig):
    mask = df['hdbscan_cluster'] == cl
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
               c=cmap_o[i % len(cmap_o)], s=20, alpha=0.9, linewidths=0,
               label=f'Sub-{cl} ({mask.sum()})')
ax.set_title(f'T1: HDBSCAN Direto\n96.3% ruído | 2 sub-clusters', fontsize=11, fontweight='bold',
             color='#B71C1C')
ax.legend(fontsize=9, markerscale=2)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

# Painel 2: T2A — UMAP+HDBSCAN global
ax = axes[1]
ax.scatter(df.loc[noise_mask, 'umap_x'], df.loc[noise_mask, 'umap_y'],
           c='#CCCCCC', s=4, alpha=0.25, linewidths=0, label=f'Ruído ({noise_mask.sum():,})')
ci = 0
for cl in [x for x in sorted(set(labels_global)) if x != -1]:
    mask = df['umap_hdbscan_global'] == cl
    ax.scatter(df.loc[mask, 'umap_x'], df.loc[mask, 'umap_y'],
               c=[cmap_g[ci]], s=12, alpha=0.85, linewidths=0, label=f'Sub-{cl} ({mask.sum()})')
    ci += 1
ax.set_title(f'T2: UMAP+HDBSCAN Global\n{pct_ruido_g:.1f}% ruído | {n_clusters_g} sub-clusters',
             fontsize=11, fontweight='bold', color='#E65100')
ax.legend(fontsize=9, markerscale=2)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

# Painel 3: T3 — LOF (score contínuo)
ax = axes[2]
sc = ax.scatter(
    df['umap_x'], df['umap_y'],
    c=df['lof_score'],
    cmap='YlOrRd',
    norm=mcolors.PowerNorm(gamma=0.4, vmin=df['lof_score'].quantile(0.01),
                            vmax=df['lof_score'].quantile(0.995)),
    s=8, alpha=0.7, linewidths=0,
)
plt.colorbar(sc, ax=ax, label='LOF Score')
ax.set_title(f'T3: LOF\n100% classificados | Score contínuo',
             fontsize=11, fontweight='bold', color='#1B5E20')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.grid(alpha=0.15)

plt.suptitle('Comparação das Três Tentativas — Espaço UMAP 2D', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_comparacao_tentativas.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: fig_comparacao_tentativas.png')

---
## Seção 5 — Decisão e Justificativa

### Por que descartamos o HDBSCAN original (T1)
O HDBSCAN aplicado diretamente às 20 features produziu **96.3% de ruído**, resultado da maldição da dimensionalidade: em espaços de alta dimensão, todas as distâncias se equalizam e não há variação de densidade suficiente para o algoritmo operar. Os 2 sub-clusters encontrados (C0 e C3) tinham valor analítico real, mas a abordagem é insustentável como método geral.

### Por que o UMAP+HDBSCAN (T2) melhorou mas tem limitações
A redução dimensional com UMAP resolve o problema de dimensionalidade e **reduz significativamente a taxa de ruído**. O UMAP+HDBSCAN global identifica sub-grupos coesos no espaço projetado. Porém:
- Os sub-clusters encontrados **podem não corresponder a agrupamentos interpretáveis** no espaço original das features
- A abordagem requer calibração dupla (parâmetros UMAP + HDBSCAN), aumentando a complexidade
- Para uso regulatório, sub-grupos discretos são menos úteis que rankings contínuos

### Por que o LOF (T3) é a escolha recomendada para uso regulatório
O LOF atende melhor ao **objetivo central do projeto**: identificar municípios que merecem atenção diferenciada dentro do seu perfil esperado.
- **Score contínuo** → permite rankear municípios em vez de classificar binariamente
- **Operação por cluster** → o município é avaliado contra seus *pares*, não contra toda a base
- **Interpretabilidade** → um score LOF alto em C3 significa "município mais atípico dentro dos já críticos Norte/Amazônicos", com implicação regulatória direta
- **Robustez** → n_neighbors é o único hiperparâmetro sensível; contamination define apenas o threshold de corte

### Recomendação final

| Necessidade analítica | Abordagem recomendada |
|---|---|
| Visualização exploratória da estrutura global | **UMAP 2D** (já gerado) |
| Identificar sub-grupos naturais dentro de clusters | **UMAP+HDBSCAN** (T2) |
| Ranking de municípios excepcionais para uso regulatório | **LOF** (T3) ✓ |
| Detectar anomalias pontuais em monitoramento contínuo | **LOF** (T3) ✓ |

> **Decisão:** O **LOF** é adotado como método principal para identificação de municípios excepcionais.  
> O UMAP (2D) é mantido como ferramenta de visualização.  
> O HDBSCAN (T2) pode ser usado exploratoramente, mas não como produto analítico primário.

In [ ]:
# Bloco E — Exportação dos artefatos

# 1. Parquet enriquecido com UMAP e LOF
arq_v2 = Path('rqual_2022_clusterizado_v2.parquet')
df.to_parquet(arq_v2, index=False)
print(f'Salvo: {arq_v2} | Shape: {df.shape}')
print(f'  Colunas novas: umap_x, umap_y, umap_hdbscan_global, umap_hdbscan_local, lof_score, lof_outlier')

# 2. CSV dos municípios excepcionais (LOF), ordenado por cluster e score
cols_export = (
    ['cod_mun', 'nome_mun_rqual', 'uf_rqual', 'kmeans_cluster', 'lof_score', 'lof_outlier']
    + IND_COLS + SOC_COLS
)
cols_export = [c for c in cols_export if c in df.columns]

df_excepcionais = (
    df[df['lof_outlier']]
    .copy()
    .sort_values(['kmeans_cluster', 'lof_score'], ascending=[True, False])
)[cols_export]

arq_exc = Path('municipios_excepcionais_lof.csv')
df_excepcionais.to_csv(arq_exc, index=False)
print(f'\nSalvo: {arq_exc} | {len(df_excepcionais)} municípios excepcionais')
print()
print('Distribuição por cluster:')
display(
    df_excepcionais.groupby('kmeans_cluster')
    .agg(n_excepcionais=('cod_mun', 'count'),
         lof_score_medio=('lof_score', 'mean'),
         lof_score_max=('lof_score', 'max'))
    .round(3)
    .rename(index=NOMES_CURTOS)
)

# 3. Sumário da decisão metodológica
print()
print('=' * 55)
print('SUMÁRIO METODOLÓGICO')
print('=' * 55)
print(f'Tentativa 1 (HDBSCAN 20D)  → DESCARTADO: {pct_ruido_orig:.1f}% ruído')
print(f'Tentativa 2 (UMAP+HDBSCAN) → COMPLEMENTAR: exploratório')
print(f'Tentativa 3 (LOF)           → ADOTADO: produto regulatório')
print()
print(f'Municípios excepcionais identificados: {len(df_excepcionais)}')
print(f'Arquivos gerados: {arq_v2.name}, {arq_exc.name}')